# 2Wiki Inference with CLaRa
Brief notebook to run inference and save predictions.

In [3]:
from pathlib import Path
import json
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModel

c:\Users\ADMIN\anaconda3\envs\clara\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
ROOT = Path.cwd()
DATA_PATH = ROOT / "2WikiMultihopQA" / "dev.parquet"
MODEL_PATH = ROOT /"CLaRa-7B-Instruct" / "compression-128"
OUT_PATH = ROOT / "results" / "2wiki_predictions_dev.jsonl"
NUM_SAMPLES = 100  # None for full split
MAX_DOCS = 20
MAX_NEW_TOKENS = 64
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [5]:
import importlib.util
import subprocess
import sys

has_fastparquet = importlib.util.find_spec("fastparquet") is not None
has_pyarrow = importlib.util.find_spec("pyarrow") is not None

if not has_fastparquet:
    print("Installing fastparquet (preferred parquet engine)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "fastparquet"])
    has_fastparquet = True

if not has_pyarrow:
    print("Installing pyarrow (fallback parquet engine)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pyarrow"])
    has_pyarrow = True

print(f"fastparquet: {has_fastparquet}, pyarrow: {has_pyarrow}")
print("If a package was installed just now, restart the kernel once before rerunning.")

fastparquet: True, pyarrow: True
If a package was installed just now, restart the kernel once before rerunning.


In [6]:
def read_parquet_robust(path):
    errors = {}

    try:
        return pd.read_parquet(path, engine="fastparquet")
    except Exception as exc:
        errors["fastparquet"] = repr(exc)

    try:
        return pd.read_parquet(path, engine="pyarrow")
    except Exception as exc:
        errors["pyarrow"] = repr(exc)

    raise RuntimeError(f"Failed to read parquet with all engines: {errors}")



df = read_parquet_robust(DATA_PATH)
if NUM_SAMPLES is not None:
    df = df.head(NUM_SAMPLES).copy()

def context_to_docs(context_raw):
    context = json.loads(context_raw) if isinstance(context_raw, str) else context_raw
    docs = []
    for item in context:
        if isinstance(item, dict):
            title = item.get("title", "")
            content = item.get("content", [])
        else:
            title, content = item
        text = " ".join(content) if isinstance(content, list) else str(content)
        docs.append((f"{title}: {text}").strip())
    return docs

records = []
for row in df.to_dict(orient="records"):
    records.append({
        "id": row.get("_id", ""),
        "question": row["question"],
        "gold_answer": row.get("answer", ""),
        "docs": context_to_docs(row["context"]),
    })

len(records)

100

In [ ]:
# Resolve local model path first, then try AutoModel, then CLaRa fallback.
from pathlib import Path
import sys
import importlib.util

def load_clara_class_from_file(repo_root: Path):
    model_file = repo_root / "ml-clara" / "openrlhf" / "models" / "modeling_clara.py"
    if not model_file.exists():
        raise FileNotFoundError(f"Cannot find modeling_clara.py at: {model_file}")

    spec = importlib.util.spec_from_file_location("local_modeling_clara", str(model_file))
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module.CLaRa


try:
    model = AutoModel.from_pretrained(str(MODEL_PATH), trust_remote_code=True).to(DEVICE)
except Exception as auto_exc:
    CLaRa = load_clara_class_from_file(ROOT)

    try:
        model = CLaRa.from_pretrained(str(MODEL_PATH), pure_inference=True).to(DEVICE)
    except TypeError:
        model = CLaRa.from_pretrained(str(MODEL_PATH)).to(DEVICE)

    print(f"AutoModel failed, loaded with direct CLaRa fallback: {auto_exc}")

model.eval()
gen_top_k = int(getattr(model, "generation_top_k", 5))

predictions = []
for sample in tqdm(records):
    q = sample["question"]
    docs = sample["docs"][:MAX_DOCS]
    if not docs:
        docs = [""]
    if len(docs) < gen_top_k:
        docs = docs + [docs[-1]] * (gen_top_k - len(docs))

    try:
        if "query_reasoner_adapter" in getattr(model, "adapter_keys", []):
            out, topk_idx = model.generate_from_questions(
                questions=[q],
                documents=[docs],
                max_new_tokens=MAX_NEW_TOKENS,
            )
            pred = out[0]
            topk = topk_idx[0].tolist() if hasattr(topk_idx[0], "tolist") else topk_idx[0]
        else:
            out = model.generate_from_text(
                questions=[q],
                documents=[docs],
                max_new_tokens=MAX_NEW_TOKENS,
            )
            pred = out[0]
            topk = None
    except Exception as exc:
        pred = f"__ERROR__: {exc}"
        topk = None

    predictions.append({
        "id": sample["id"],
        "question": q,
        "gold_answer": sample["gold_answer"],
        "prediction": pred,
        "topk_idx": topk,
    })

with OUT_PATH.open("w", encoding="utf-8") as f:
    for row in predictions:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"Saved {len(predictions)} rows to {OUT_PATH}")
predictions[0] if predictions else {}

Initializing model from trained checkpoint: CLaRaConfig {
  "_attn_implementation_autoset": true,
  "ae_mode": "token",
  "attn_implementation": null,
  "auto_map": {
    "AutoConfig": "modeling_clara.CLaRaConfig",
    "AutoModel": "modeling_clara.CLaRa"
  },
  "compr_base_model_name": "mistralai/Mistral-7B-Instruct-v0.2",
  "compr_every_n_layer": null,
  "compr_linear_type": "concat",
  "compr_mlp_hidden_dim": 8096,
  "compr_model_name": null,
  "compr_n_layers": 5,
  "compr_rate": 128,
  "compr_rms_norm": false,
  "compr_use_mlp": false,
  "decoder_model_name": "mistralai/Mistral-7B-Instruct-v0.2",
  "device_map": null,
  "different_mem_tokens": true,
  "doc_max_length": 256,
  "generation_top_k": 5,
  "kbtc_training": false,
  "load_adapters": false,
  "load_pretrained_checkpoint": false,
  "lora": true,
  "lora_compressor": false,
  "lora_r": 16,
  "lora_r_compressor": 16,
  "max_new_tokens": 128,
  "model_type": "CLaRa",
  "optimize_mem_tokens": true,
  "pad_token_id": 2,
  "pure_

`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]